[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pradeepvaka/llm-inference-90day/blob/master/notebooks/day24-tool-calling.ipynb)

# Day 24 — Tool Calling

**Big idea:** the model never calls anything — it *proposes* calls as text; your code is the dispatcher that executes and feeds results back. Latency multiplies by rounds, and prefill is re-paid every round.

**Run plan (30–60 min):** Cells 4–12 are pure Python and run on CPU in minutes. Cell 14 is the optional live vLLM demo and needs a T4 (Runtime → Change runtime type → T4 GPU); on CPU it prints a skip message. No downloads over ~2 GB.


In [ ]:
# Cell 2: one-time install (only cell that installs).
%pip install --quiet torch
import torch
print(f"torch {torch.__version__}  cuda available: {torch.cuda.is_available()}")
# Expected (CPU): torch 2.x  cuda available: False
# Expected (T4):  torch 2.x  cuda available: True


## 1. Tool schemas and the prompt tax

Three tools as OpenAI-style JSON schemas. The schemas ride in *every* request's prompt, so we estimate their token cost (JSON characters ÷ 4 ≈ tokens) before running anything.


In [ ]:
# Cell 4: define the 3 tools; estimate schema tokens as len(json) / 4.
import json

TOOLS = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get the current weather for a city. Returns temperature, conditions, and humidity.",
      "parameters": {
        "type": "object",
        "properties": {
          "city": {"type": "string", "description": "City name, e.g. Paris or San Francisco"},
          "units": {"type": "string", "enum": ["celsius", "fahrenheit"], "description": "Temperature unit"}
        },
        "required": ["city"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "convert_units",
      "description": "Convert a numeric value from one unit to another, e.g. celsius to fahrenheit or km to miles.",
      "parameters": {
        "type": "object",
        "properties": {
          "value": {"type": "number", "description": "The numeric value to convert"},
          "from_unit": {"type": "string", "description": "Source unit, e.g. celsius"},
          "to_unit": {"type": "string", "description": "Target unit, e.g. fahrenheit"}
        },
        "required": ["value", "from_unit", "to_unit"]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "calculator",
      "description": "Evaluate a simple arithmetic expression and return the result.",
      "parameters": {
        "type": "object",
        "properties": {
          "expression": {"type": "string", "description": "Arithmetic expression, e.g. 22 * 9 / 5 + 32"}
        },
        "required": ["expression"]
      }
    }
  },
]

total = 0
for t in TOOLS:
    n = len(json.dumps(t)) / 4
    total += n
    print(f"{t['function']['name']} ~{n:.0f} tokens")
print(f"total ~{total:.0f} tokens")
# Expected:
# get_weather ~104 tokens
# convert_units ~124 tokens
# calculator ~78 tokens
# total ~306 tokens


The lab schemas above are lean (~100 tokens each). Production schemas with nested objects and examples run ~300 tokens each — the number the packet's tax table uses. Measure both scales below.

In [ ]:
# Cell 6: prompt-tax table at ~300 tokens/tool, prefill ~23,333 tokens/s.
THR = 3500 / 0.15  # tokens/s implied by 3,500 tokens in 150 ms

for n in (1, 5, 10, 20, 50):
    toks = n * 300
    print(f"{n:2d} tools -> {toks:5d} schema tokens, ~{toks / THR * 1000:3.0f} ms extra TTFT per round, {toks * 3:5d} schema prefills over 3 rounds")
# Expected:
#  1 tools ->   300 schema tokens, ~ 13 ms extra TTFT per round,   900 schema prefills over 3 rounds
#  5 tools ->  1500 schema tokens, ~ 64 ms extra TTFT per round,  4500 schema prefills over 3 rounds
# 10 tools ->  3000 schema tokens, ~129 ms extra TTFT per round,  9000 schema prefills over 3 rounds
# 20 tools ->  6000 schema tokens, ~257 ms extra TTFT per round, 18000 schema prefills over 3 rounds
# 50 tools -> 15000 schema tokens, ~643 ms extra TTFT per round, 45000 schema prefills over 3 rounds


## 2. The loop: fake model, real executor (CPU)

A scripted fake model stands in for the LLM so the accounting is exact and reproducible. The executor is real: `get_weather` returns a stub 22 °C, `convert_units` computes 22 °C → 71.6 °F, `calculator` safely evaluates arithmetic. Prefill starts at 3,500 (300 system + 3,000 schemas + 200 user/history) and grows by (call tokens + result tokens) each round. TPOT = 40 ms, tool = 200 ms, prefill = 150 ms/round.

In [ ]:
# Cell 8: executor + scripted loop for "weather in Paris, in Fahrenheit?"
import ast, operator

WEATHER_STUB = {"Paris": {"temp_c": 22.0, "conditions": "clear", "humidity": 0.40},
                "London": {"temp_c": 15.0, "conditions": "overcast", "humidity": 0.65}}

def get_weather(city, units="celsius"):
    w = WEATHER_STUB[city]
    temp = w["temp_c"] if units == "celsius" else w["temp_c"] * 9 / 5 + 32
    return {"city": city, "temp": temp, "unit": units, "conditions": w["conditions"], "humidity": w["humidity"]}

def convert_units(value, from_unit, to_unit):
    if from_unit == "celsius" and to_unit == "fahrenheit":
        return {"value": value * 9 / 5 + 32, "unit": "fahrenheit"}
    if from_unit == "fahrenheit" and to_unit == "celsius":
        return {"value": (value - 32) * 5 / 9, "unit": "celsius"}
    raise ValueError(f"unsupported conversion {from_unit} -> {to_unit}")

_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
        ast.Div: operator.truediv, ast.USub: operator.neg}
def _ev(node):
    if isinstance(node, ast.Expression):
        return _ev(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_ev(node.left), _ev(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _OPS:
        return _OPS[type(node.op)](_ev(node.operand))
    raise ValueError("unsupported expression")

def calculator(expression):
    return {"value": _ev(ast.parse(expression, mode="eval"))}

EXECUTORS = {"get_weather": get_weather, "convert_units": convert_units, "calculator": calculator}

SCRIPT = [
    {"name": "get_weather", "arguments": {"city": "Paris", "units": "celsius"}, "decode_tokens": 60, "result_tokens": 140},
    {"name": "convert_units", "arguments": {"value": 22.0, "from_unit": "celsius", "to_unit": "fahrenheit"}, "decode_tokens": 40, "result_tokens": 60},
]

TPOT_MS, TOOL_MS, PREFILL_MS = 40, 200, 150
prefill = 3500
tot_prefill = tot_decode = tot_ms = 0
for r, call in enumerate(SCRIPT, start=1):
    result = EXECUTORS[call["name"]](**call["arguments"])
    ms = PREFILL_MS + call["decode_tokens"] * TPOT_MS + TOOL_MS
    print(f"round {r}: prefill={prefill} decode={call['decode_tokens']} tool={TOOL_MS}ms -> {call['name']} = {result}")
    tot_prefill += prefill; tot_decode += call["decode_tokens"]; tot_ms += ms
    prefill += call["decode_tokens"] + call["result_tokens"]

check = calculator("22 * 9 / 5 + 32")
print(f"round 3: prefill={prefill} decode=80 tool=0ms -> final answer (calculator check: {check})")
tot_prefill += prefill; tot_decode += 80; tot_ms += PREFILL_MS + 80 * TPOT_MS
print(f"TOTAL: prefill={tot_prefill} decode={tot_decode} tokens={tot_prefill + tot_decode} latency={tot_ms} ms")
assert (tot_prefill, tot_decode, tot_prefill + tot_decode, tot_ms) == (11000, 180, 11180, 8050)
print("assertions passed: 11000 prefill + 180 decode = 11180 tokens, 8050 ms")
# Expected:
# round 1: prefill=3500 decode=60 tool=200ms -> get_weather = {'city': 'Paris', 'temp': 22.0, 'unit': 'celsius', 'conditions': 'clear', 'humidity': 0.4}
# round 2: prefill=3700 decode=40 tool=200ms -> convert_units = {'value': 71.6, 'unit': 'fahrenheit'}
# round 3: prefill=3800 decode=80 tool=0ms -> final answer (calculator check: {'value': 71.6})
# TOTAL: prefill=11000 decode=180 tokens=11180 latency=8050 ms
# assertions passed: 11000 prefill + 180 decode = 11180 tokens, 8050 ms


## 3. Serial vs parallel: the dependency graph decides

Paris-then-convert is *dependent* (serial, unavoidable). Paris-and-London weather is *independent*: both calls fit in one round and execute concurrently. Verified saving at 200 ms tools is modest — one prefill (150 ms) plus tool serialization (200 ms) — and grows with tool latency, not decode speed.

In [ ]:
# Cell 10: serial (3 rounds) vs parallel (2 rounds) for two independent weather calls.
def round_ms(decode_toks, tool_ms):
    return PREFILL_MS + decode_toks * TPOT_MS + tool_ms

# Serial: call Paris (60), call London (60), answer (80); tools serialized 200+200 ms.
serial = round_ms(60, 200) + round_ms(60, 200) + round_ms(80, 0)
# Parallel: both calls in round 1 (120 decode tokens), tools concurrent (200 ms once), answer round 2.
parallel = round_ms(120, 200) + round_ms(80, 0)
print(f"serial:   3 rounds, {serial} ms")
print(f"parallel: 2 rounds, {parallel} ms")
print(f"saved: {serial - parallel} ms")
assert (serial, parallel, serial - parallel) == (8850, 8500, 350)
print("assertions passed: serial 8850 ms, parallel 8500 ms, saved 350 ms")
# Expected:
# serial:   3 rounds, 8850 ms
# parallel: 2 rounds, 8500 ms
# saved: 350 ms
# assertions passed: serial 8850 ms, parallel 8500 ms, saved 350 ms


## 4. Errors are tool results too

Break `get_weather` so it raises on the first call. The loop catches the exception, appends it as a `role: "tool"` error payload, and lets the (scripted) model retry. One failure costs exactly one extra round. Never append an empty success for a failed call.

In [ ]:
# Cell 12: flaky weather tool — fails once, then succeeds; loop recovers in 4 rounds total.
attempts = {"n": 0}

def flaky_weather(city, units="celsius"):
    attempts["n"] += 1
    if attempts["n"] == 1:
        raise TimeoutError("weather API timeout after 200 ms")
    return get_weather(city, units)

messages = [{"role": "user", "content": "Weather in Paris in Fahrenheit?"}]
rounds = 0
# Scripted recovery: round 1 errors, round 2 retries weather, round 3 converts, round 4 answers.
plan = ["weather", "weather", "convert", "answer"]
for step in plan:
    rounds += 1
    if step == "weather":
        try:
            out = flaky_weather("Paris", "celsius")
            messages.append({"role": "tool", "content": str(out)})
            print(f"round {rounds}: tool ok -> {out}")
        except Exception as e:
            messages.append({"role": "tool", "content": str({"error": str(e)})})
            print(f"round {rounds}: error fed back -> {{'error': '{e}'}}")
    elif step == "convert":
        out = convert_units(22.0, "celsius", "fahrenheit")
        messages.append({"role": "tool", "content": str(out)})
        print(f"round {rounds}: convert -> {out}")
    else:
        print(f"round {rounds}: final answer: Paris is 71.6 degrees Fahrenheit (clear).")
print(f"recovered in {rounds} rounds total (clean run: 3; broken tool: +1)")
assert rounds == 4 and attempts["n"] == 2
print("assertions passed: recovered in 4 rounds, 2 weather attempts")
# Expected:
# round 1: error fed back -> {'error': 'weather API timeout after 200 ms'}
# round 2: tool ok -> {'city': 'Paris', 'temp': 22.0, 'unit': 'celsius', 'conditions': 'clear', 'humidity': 0.4}
# round 3: convert -> {'value': 71.6, 'unit': 'fahrenheit'}
# round 4: final answer: Paris is 71.6 degrees Fahrenheit (clear).
# recovered in 4 rounds total (clean run: 3; broken tool: +1)
# assertions passed: recovered in 4 rounds, 2 weather attempts


## 5. Live demo on T4 (optional): real model, same loop

Loads Qwen2.5-1.5B-Instruct in vLLM and runs the *same* executor loop against the real model's tool calls. SmolLM-class models do not emit reliable tool calls — an instruct model ≥1B is the requirement. On CPU this cell prints a skip message; rerun on a T4 to see real per-round accounting (wording varies; round count and 71.6 are the check). Keep `max_rounds=5` and treat the printed table as your What-to-measure numbers.

In [ ]:
# Cell 14: live vLLM tool loop (T4 only; guarded so CPU runs skip cleanly).
import torch

if not torch.cuda.is_available():
    print("CPU-only: skipping live vLLM demo. Rerun on a T4 to load Qwen2.5-1.5B-Instruct.")
    print("Expected on T4: 3 rounds, final answer mentions 71.6 (wording varies).")
else:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "vllm"], check=True)
    from vllm import LLM, SamplingParams
    llm = LLM(model="Qwen/Qwen2.5-1.5B-Instruct", max_model_len=4096, gpu_memory_utilization=0.85)
    print("vLLM ready on cuda: Qwen/Qwen2.5-1.5B-Instruct")
    print("Now run your Day 15-style loop: send TOOLS + prompt, parse tool_calls, dispatch via EXECUTORS,")
    print("append role=tool results, repeat (max 5 rounds). Log prefill/decode tokens + ms per round.")
    print("Expected: 3 rounds, final answer mentions 71.6 (wording varies).")
# Expected (CPU):
# CPU-only: skipping live vLLM demo. Rerun on a T4 to load Qwen2.5-1.5B-Instruct.
# Expected on T4: 3 rounds, final answer mentions 71.6 (wording varies).


## Wrap-up: fill in and self-check

Record your numbers, then answer cold (answers in the packet):

| Metric | Your number |
|---|---|
| Schema tokens, 3-tool lab set | |
| Prefill per round (1–3) | / / |
| Decode per round (1–3) | / / |
| Total latency, Paris task (sim) | |
| Total tokens processed | |
| Rounds with broken tool | |
| Parallel demo: saved ms | |

**Checkpoints:** (1) Who executes the tool — model or your code? (2) 3 serial rounds at 4 s/round — total latency? (3) 10 tools at ~300 tokens — per-round prefill tax?

**Tomorrow (Day 25):** prefix/prompt caching — those schema tokens become reusable KV blocks, so round 2 stops re-paying round 1's prefill.

In [ ]:
# Cell 16: final verification — every packet number, asserted.
assert convert_units(22.0, "celsius", "fahrenheit")["value"] == 71.6
assert calculator("22 * 9 / 5 + 32")["value"] == 71.6
print("all checks passed")
print("Next: Day 25 prefix caching reclaims the 11,000-token prefill bill you just measured.")
# Expected:
# all checks passed
# Next: Day 25 prefix caching reclaims the 11,000-token prefill bill you just measured.
